# Evaluación Experimental del Buscador Semántico

**Responsable:** Persona 3 (Renato) — Especialista en Análisis Experimental y Visualización

Este notebook ejecuta el pipeline completo del buscador semántico, genera las visualizaciones
y analiza los resultados experimentales.

---

## 1. Imports y Configuración

In [ ]:
import sys
import os
import numpy as np

# Asegurar que el directorio raíz esté en el path
directorio_raiz = os.path.abspath(os.path.join(os.getcwd(), '..'))
if directorio_raiz not in sys.path:
    sys.path.insert(0, directorio_raiz)

# Configurar matplotlib para mostrar gráficos inline
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Importar módulos del proyecto
from src.preprocesamiento import cargar_documentos, preprocesar_corpus
from src.vectorizacion import construir_vocabulario, crear_matriz_documento_termino, obtener_vocabulario_inverso
from src.similitud import similitud_coseno, calcular_producto_punto, calcular_norma
from src.buscador import BuscadorSemantico
from src.visualizacion import (
    plot_matriz_similitud,
    plot_pca_documentos,
    plot_frecuencia_terminos,
    plot_resultados_busqueda
)

print('Imports exitosos. Listo para ejecutar.')

## 2. Cargar y Preprocesar el Corpus

In [ ]:
# Cargar documentos desde el corpus
ruta_corpus = os.path.join(directorio_raiz, 'data', 'documentos', 'corpus.txt')
documentos = cargar_documentos(ruta_corpus)

print(f'Documentos cargados: {len(documentos)}')
print('\n--- Vista previa de los documentos ---')
for i, doc in enumerate(documentos):
    print(f'  Doc {i+1}: {doc[:100]}...')

In [ ]:
# Preprocesar: limpiar, tokenizar, remover stopwords
corpus_procesado = preprocesar_corpus(documentos)

print('--- Tokens por documento (tras preprocesamiento) ---')
for i, tokens in enumerate(corpus_procesado):
    print(f'  Doc {i+1} ({len(tokens)} tokens): {tokens[:8]}...')

## 3. Construir Vocabulario y Matriz Documento-Término

In [ ]:
# Construir vocabulario
vocabulario = construir_vocabulario(corpus_procesado)
print(f'Tamaño del vocabulario: {len(vocabulario)} términos únicos')
print(f'Primeros 20 términos: {list(vocabulario.keys())[:20]}')

# Crear la Matriz Documento-Término
matriz_dt = crear_matriz_documento_termino(corpus_procesado, vocabulario)
print(f'\nDimensiones de la Matriz Documento-Término: {matriz_dt.shape}')
print(f'  → {matriz_dt.shape[0]} documentos (filas)')
print(f'  → {matriz_dt.shape[1]} términos (columnas)')

# Calcular sparsidad
ceros = np.count_nonzero(matriz_dt == 0)
total = matriz_dt.size
sparsidad = (ceros / total) * 100
print(f'\nSparsidad de la matriz: {sparsidad:.1f}%')
print(f'  → {ceros} de {total} entradas son cero')

## 4. Calcular Matriz de Similitud entre Documentos

In [ ]:
# Crear etiquetas para los documentos
etiquetas = [f'Doc {i+1}' for i in range(len(documentos))]

# Inicializar el buscador
buscador = BuscadorSemantico(matriz_dt, vocabulario, documentos, etiquetas)

# Calcular la matriz de similitud coseno entre todos los documentos
matriz_similitud = buscador.calcular_matriz_similitud()

print('Matriz de Similitud Coseno (12x12):')
print(np.round(matriz_similitud, 3))

## 5. Heatmap de Similitud entre Documentos

In [ ]:
# Generar y guardar el heatmap
plot_matriz_similitud(matriz_similitud, etiquetas, guardar=True, mostrar=False)

# También mostrarlo inline en el notebook
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matriz_similitud, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=etiquetas, yticklabels=etiquetas,
    vmin=0, vmax=1, square=True, linewidths=0.5,
    cbar_kws={'label': 'Similitud Coseno'}, ax=ax
)
ax.set_title('Matriz de Similitud Coseno entre Documentos', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

### Interpretación del Heatmap

- La diagonal principal muestra **1.00** en todas las celdas: cada documento es idéntico a sí mismo.
- Las celdas con colores más cálidos (rojo/naranja) indican **mayor similitud semántica**.
- Las celdas en amarillo claro indican **baja o nula similitud**.
- La matriz es **simétrica**: la similitud de Doc A con Doc B es igual a la de Doc B con Doc A.

## 6. Reducción Dimensional con PCA (Scatter Plot)

In [ ]:
# PCA sin consulta
plot_pca_documentos(matriz_dt, etiquetas, guardar=True, mostrar=False)

# PCA inline con explicación
pca = PCA(n_components=2)
datos_2d = pca.fit_transform(matriz_dt)
var_explicada = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(datos_2d[:, 0], datos_2d[:, 1], c='#2196F3', s=120, alpha=0.8,
           edgecolors='white', linewidth=1.5, zorder=3)

for i, etq in enumerate(etiquetas):
    ax.annotate(etq, (datos_2d[i, 0], datos_2d[i, 1]),
                textcoords='offset points', xytext=(8, 8), fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='gray', alpha=0.7))

ax.set_xlabel(f'PC1 ({var_explicada[0]:.1f}% varianza)', fontsize=11)
ax.set_ylabel(f'PC2 ({var_explicada[1]:.1f}% varianza)', fontsize=11)
ax.set_title('Reducción Dimensional PCA — Documentos en 2D', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print(f'\nVarianza explicada por PC1: {var_explicada[0]:.1f}%')
print(f'Varianza explicada por PC2: {var_explicada[1]:.1f}%')
print(f'Varianza total explicada: {sum(var_explicada):.1f}%')

### Interpretación del PCA

- PCA reduce las **154 dimensiones** del espacio vectorial a solo **2 dimensiones** preservando la máxima varianza posible.
- Documentos **cercanos** en el gráfico comparten más vocabulario (y por tanto, más contenido semántico).
- Los **ejes** representan combinaciones lineales de los términos originales que mejor separan los datos.
- La varianza explicada indica **qué porcentaje de la información** original se conserva en la proyección 2D.

## 7. Frecuencia de Términos del Corpus

In [ ]:
# Generar gráfico de frecuencia
plot_frecuencia_terminos(matriz_dt, vocabulario, top_n=15, guardar=True, mostrar=False)

# Mostrar inline
frecuencias_totales = np.sum(matriz_dt, axis=0)
vocab_inv = obtener_vocabulario_inverso(vocabulario)
terminos_freq = [(vocab_inv[i], frecuencias_totales[i]) for i in range(len(frecuencias_totales))]
terminos_freq.sort(key=lambda x: x[1], reverse=True)

print('Top 15 Términos Más Frecuentes:')
for i, (palabra, freq) in enumerate(terminos_freq[:15], 1):
    print(f'  {i:2d}. {palabra:20s} → {int(freq)} apariciones')

## 8. Búsquedas Experimentales

In [ ]:
# Definir consultas experimentales
consultas = [
    'inteligencia artificial aprendizaje',
    'seguridad informática redes protección',
    'programación software desarrollo',
    'datos bases consultas información',
    'robots autónomos ingeniería',
]

print('=' * 70)
print('RESULTADOS DE BÚSQUEDAS EXPERIMENTALES')
print('=' * 70)

for consulta in consultas:
    resultados = buscador.buscar(consulta, top_n=5)
    print(f'\n🔍 Consulta: "{consulta}"')
    print('-' * 50)
    for i, r in enumerate(resultados, 1):
        print(f'  {i}. [{r["etiqueta"]}] Similitud: {r["similitud"]:.4f}')
    print()

## 9. Gráficos de Resultados de Búsqueda

In [ ]:
# Generar gráficos individuales por búsqueda
for i, consulta in enumerate(consultas, 1):
    resultados = buscador.buscar(consulta, top_n=5)
    plot_resultados_busqueda(resultados, consulta, nombre_archivo=f'busqueda_{i}', guardar=True)

# PCA con la primera consulta como ejemplo
vector_q = buscador.vectorizar_consulta(consultas[0])
plot_pca_documentos(matriz_dt, etiquetas, consulta_vector=vector_q,
                    consulta_texto=consultas[0], guardar=True)

print('\nTodos los gráficos de búsqueda generados y guardados en graficos/')

## 10. Análisis del Impacto del Tamaño del Vocabulario

Experimentamos reduciendo progresivamente el vocabulario para observar cómo afecta la calidad de las búsquedas.

In [ ]:
# Analizar impacto del tamaño del vocabulario
consulta_test = 'inteligencia artificial aprendizaje'

# Obtener frecuencias para poder filtrar por las más comunes
freq_totales = np.sum(matriz_dt, axis=0)
vocab_inv = obtener_vocabulario_inverso(vocabulario)

# Ordenar palabras por frecuencia descendente
indices_ordenados = np.argsort(freq_totales)[::-1]

# Probar con vocabularios de diferentes tamaños
tamanos = [10, 25, 50, 100, len(vocabulario)]
resultados_por_tamano = []

print(f'Consulta de prueba: "{consulta_test}"')
print('=' * 70)

for tam in tamanos:
    # Crear sub-vocabulario con las 'tam' palabras más frecuentes
    top_indices = indices_ordenados[:tam]
    sub_vocab = {vocab_inv[idx]: nuevo_idx for nuevo_idx, idx in enumerate(top_indices)
                 if idx in vocab_inv}

    # Reconstruir matriz con sub-vocabulario
    sub_matriz = crear_matriz_documento_termino(corpus_procesado, sub_vocab)

    # Crear buscador con sub-vocabulario
    sub_buscador = BuscadorSemantico(sub_matriz, sub_vocab, documentos, etiquetas)
    resultados = sub_buscador.buscar(consulta_test, top_n=3)

    # Sparsidad del sub-vocabulario
    ceros_sub = np.count_nonzero(sub_matriz == 0)
    sparsidad_sub = (ceros_sub / sub_matriz.size) * 100

    print(f'\nVocabulario: {tam} términos | Sparsidad: {sparsidad_sub:.1f}%')
    print(f'  Dimensiones de la matriz: {sub_matriz.shape}')
    for r in resultados:
        print(f'  → [{r["etiqueta"]}] Similitud: {r["similitud"]:.4f}')

    resultados_por_tamano.append({
        'tamano': tam,
        'sparsidad': sparsidad_sub,
        'top1_similitud': resultados[0]['similitud'] if resultados else 0,
        'top1_doc': resultados[0]['etiqueta'] if resultados else 'N/A'
    })

In [ ]:
# Gráfico del impacto del vocabulario
tamanos_plot = [r['tamano'] for r in resultados_por_tamano]
similitudes_plot = [r['top1_similitud'] for r in resultados_por_tamano]
sparsidades_plot = [r['sparsidad'] for r in resultados_por_tamano]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Similitud vs Tamaño del vocabulario
ax1.plot(tamanos_plot, similitudes_plot, 'o-', color='#2196F3', linewidth=2, markersize=8)
ax1.set_xlabel('Tamaño del Vocabulario', fontsize=11)
ax1.set_ylabel('Similitud Coseno (Top 1)', fontsize=11)
ax1.set_title('Impacto del Vocabulario en la Similitud', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')

# Gráfico 2: Sparsidad vs Tamaño del vocabulario
ax2.plot(tamanos_plot, sparsidades_plot, 's-', color='#F44336', linewidth=2, markersize=8)
ax2.set_xlabel('Tamaño del Vocabulario', fontsize=11)
ax2.set_ylabel('Sparsidad (%)', fontsize=11)
ax2.set_title('Sparsidad de la Matriz vs Vocabulario', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(directorio_raiz, 'graficos', 'impacto_vocabulario.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n[✓] Gráfico de impacto del vocabulario guardado.')

### Conclusiones sobre el Tamaño del Vocabulario

1. **Vocabularios muy pequeños** (10-25 términos) pierden información discriminativa: las similitudes son menos precisas porque muchos documentos comparten las mismas palabras frecuentes.
2. **A medida que crece el vocabulario**, la representación se vuelve más rica y las similitudes más significativas.
3. **La sparsidad aumenta** con el vocabulario: más columnas = más ceros, porque cada documento solo usa un subconjunto del vocabulario total.
4. **Trade-off**: un vocabulario muy grande puede introducir ruido (palabras que aparecen solo una vez), mientras que uno muy pequeño pierde matices semánticos.

---

## Resumen de Gráficos Generados

| Gráfico | Archivo | Descripción |
|---------|---------|-------------|
| Heatmap de Similitud | `graficos/heatmap_similitud.png` | Mapa de calor de la matriz de similitud coseno |
| PCA Scatter | `graficos/pca_documentos.png` | Documentos proyectados en 2D con PCA |
| Frecuencia de Términos | `graficos/frecuencia_terminos.png` | Top 15 palabras más frecuentes |
| Búsquedas 1-5 | `graficos/busqueda_*.png` | Resultados individuales de cada búsqueda |
| Impacto Vocabulario | `graficos/impacto_vocabulario.png` | Análisis de tamaño de vocabulario vs rendimiento |